# SentinelAI — 06 Model Selection, XGBoost Tuning, Final Test

المنهجية:
1. اختيار baseline الأفضل من **Validation**.
2. Hyperparameter tuning داخل Train فقط باستخدام `StratifiedKFold`.
3. تمرير `sample_weight` أثناء البحث وأثناء التدريب النهائي.
4. مقارنة tuned vs baseline على Validation.
5. اختيار واحد نهائي.
6. **فتح Test مرة واحدة فقط** للتقرير النهائي.

> لأن نتائج المشروع السابقة أظهرت XGBoost بفارق واضح، هذا الـNotebook مضبوط لتونينج XGBoost.
> إذا لم يعد XGBoost هو الأفضل على Validation بعد إعادة التشغيل، يتوقف الملف بدل ادعاء اختيار غير صحيح.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, make_scorer, classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
P = BASE_DIR / "data" / "processed"
M = BASE_DIR / "models"

X_train = np.load(P / "X_train.npy", mmap_mode="r")
X_val = np.load(P / "X_val.npy", mmap_mode="r")
X_test = np.load(P / "X_test.npy", mmap_mode="r")
y_train = np.load(P / "y_train.npy", mmap_mode="r")
y_val = np.load(P / "y_val.npy", mmap_mode="r")
y_test = np.load(P / "y_test.npy", mmap_mode="r")

label_mapping = {int(k): v for k, v in json.loads((P/"label_mapping.json").read_text(encoding="utf-8")).items()}
class_weights = {int(k): v for k, v in json.loads((P/"class_weights.json").read_text(encoding="utf-8")).items()}
le = joblib.load(M / "xgboost_label_encoder.joblib")

y_train_enc = le.transform(np.asarray(y_train))
sample_weight_train = np.array([class_weights[int(v)] for v in y_train], dtype=float)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

Train: (908327, 77) Validation: (113541, 77) Test: (113541, 77)


In [2]:
# Validation leaderboard across multiclass models only.
sup = pd.read_csv(M / "supervised_comparison.csv")
cnn = pd.read_csv(M / "cnn_comparison.csv")
leaderboard = pd.concat([sup, cnn], ignore_index=True, sort=False)
leaderboard = leaderboard.sort_values("F1_macro", ascending=False).reset_index(drop=True)

display(leaderboard[["Model", "Evaluation_Split", "Accuracy", "F1_macro"]])

best_baseline_name = str(leaderboard.iloc[0]["Model"])
print("Best validation baseline:", best_baseline_name)

if "XGBoost" not in best_baseline_name:
    raise RuntimeError(
        "XGBoost is no longer the best validation baseline. "
        "Do not continue with XGBoost-specific tuning; update the tuning strategy for "
        f"the actual winner: {best_baseline_name}"
    )

,Model,Evaluation_Split,Accuracy,F1_macro
0,Gradient Boosting (XGBoost),validation,0.999727,0.936659
1,Random Forest,validation,0.999498,0.899943
2,Logistic Regression,validation,0.934147,0.699078
3,CNN (1D Conv),validation,0.984561,0.672114


Best validation baseline: Gradient Boosting (XGBoost)


In [3]:
# Build a search subset that keeps ALL very rare classes represented enough for StratifiedKFold.
def make_search_subset(X, y, max_size=100_000, min_per_class=30, seed=42):
    rng_ = np.random.default_rng(seed)
    y = np.asarray(y)
    selected = []
    selected_mask = np.zeros(len(y), dtype=bool)

    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        take = min(len(idx), min_per_class)
        chosen = rng_.choice(idx, size=take, replace=False)
        selected.extend(chosen.tolist())
        selected_mask[chosen] = True

    remaining = max_size - len(selected)
    if remaining > 0:
        pool = np.flatnonzero(~selected_mask)
        take = min(remaining, len(pool))
        selected.extend(rng_.choice(pool, size=take, replace=False).tolist())

    selected = np.array(selected, dtype=int)
    rng_.shuffle(selected)
    return np.asarray(X[selected]), y[selected]

X_search, y_search = make_search_subset(X_train, y_train)
y_search_enc = le.transform(y_search)
sample_weight_search = np.array([class_weights[int(v)] for v in y_search], dtype=float)

print("Search subset:", X_search.shape)
print(pd.Series(y_search).value_counts().sort_index())

min_count = int(pd.Series(y_search).value_counts().min())
assert min_count >= 3, f"Need >=3 samples per class for 3-fold CV; got {min_count}"

Search subset: (100000, 77)
1     77225
2      3056
3      2990
4     10992
5      4712
6       705
7       201
8        49
9        32
10       30
11        8
Name: count, dtype: int64


In [4]:
base_estimator = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

param_distributions = {
    "n_estimators": [150, 200, 300, 400],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.08, 0.10, 0.15],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [1.0, 2.0, 5.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scorer = make_scorer(f1_score, average="macro", zero_division=0)

search = RandomizedSearchCV(
    estimator=base_estimator,
    param_distributions=param_distributions,
    n_iter=12,
    scoring=scorer,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,  # XGBoost itself uses all cores; avoid nested oversubscription.
    verbose=1,
    refit=True,
)

start = time.time()
search.fit(X_search, y_search_enc, sample_weight=sample_weight_search)
search_seconds = time.time() - start

print("Search seconds:", round(search_seconds, 1))
print("Best CV macro-F1:", search.best_score_)
print("Best params:", search.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


Search seconds: 1112.1
Best CV macro-F1: 0.8782070007866482
Best params: {'subsample': 0.8, 'reg_lambda': 5.0, 'n_estimators': 150, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 1.0}


In [5]:
# Refit tuned XGBoost on ALL training data with class-derived sample weights.
tuned = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **search.best_params_,
)

start = time.time()
tuned.fit(np.asarray(X_train), y_train_enc, sample_weight=sample_weight_train)
full_train_seconds = time.time() - start

def multiclass_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision_macro": float(p),
        "Recall_macro": float(r),
        "F1_macro": float(f1),
    }

baseline = joblib.load(M / "gradient_boosting_xgboost.joblib")

baseline_val = le.inverse_transform(baseline.predict(np.asarray(X_val)).astype(int))
tuned_val = le.inverse_transform(tuned.predict(np.asarray(X_val)).astype(int))

baseline_val_metrics = multiclass_metrics(y_val, baseline_val)
tuned_val_metrics = multiclass_metrics(y_val, tuned_val)

print("Baseline validation:", baseline_val_metrics)
print("Tuned validation:", tuned_val_metrics)

Baseline validation: {'Accuracy': 0.9997269708739575, 'Precision_macro': 0.9164416278835646, 'Recall_macro': 0.9897998570638329, 'F1_macro': 0.9366592369526422}
Tuned validation: {'Accuracy': 0.9993922900097763, 'Precision_macro': 0.8346623291548094, 'Recall_macro': 0.9998605806554521, 'F1_macro': 0.8753495936131681}


In [6]:
# Select using VALIDATION only.
if tuned_val_metrics["F1_macro"] >= baseline_val_metrics["F1_macro"]:
    selected_model = tuned
    selected_name = "XGBoost (Tuned)"
    selected_val_metrics = tuned_val_metrics
else:
    selected_model = baseline
    selected_name = "XGBoost (Baseline retained)"
    selected_val_metrics = baseline_val_metrics

print("Selected before opening test:", selected_name)
print("Selected validation metrics:", selected_val_metrics)

# FINAL TEST — this is the first use of y_test in model selection workflow.
test_pred_enc = selected_model.predict(np.asarray(X_test)).astype(int)
test_pred = le.inverse_transform(test_pred_enc)
test_metrics = multiclass_metrics(y_test, test_pred)

print("\nFINAL TEST METRICS (not used for selection)")
print(test_metrics)
print(classification_report(
    y_test, test_pred,
    labels=sorted(label_mapping),
    target_names=[label_mapping[c] for c in sorted(label_mapping)],
    zero_division=0,
))

Selected before opening test: XGBoost (Baseline retained)
Selected validation metrics: {'Accuracy': 0.9997269708739575, 'Precision_macro': 0.9164416278835646, 'Recall_macro': 0.9897998570638329, 'F1_macro': 0.9366592369526422}



FINAL TEST METRICS (not used for selection)
{'Accuracy': 0.9997974300032587, 'Precision_macro': 0.8588411612594951, 'Recall_macro': 0.9696598772124667, 'F1_macro': 0.8967175034918179}
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     88006
    Attack_2       1.00      1.00      1.00      3513
    Attack_3       1.00      1.00      1.00      3382
    Attack_4       1.00      1.00      1.00     12428
    Attack_5       1.00      1.00      1.00      5205
    Attack_6       1.00      1.00      1.00       760
    Attack_7       1.00      1.00      1.00       203
    Attack_8       1.00      1.00      1.00        30
    Attack_9       0.29      0.67      0.40         9
   Attack_10       0.67      1.00      0.80         4
   Attack_11       0.50      1.00      0.67         1

    accuracy                           1.00    113541
   macro avg       0.86      0.97      0.90    113541
weighted avg       1.00      1.00      1.00    113541



In [7]:
# Save the production model and auditable metadata.
joblib.dump(selected_model, M / "best_model.joblib")
joblib.dump(le, M / "best_model_label_encoder.joblib")

info = {
    "methodology_version": 2,
    "selected_model": selected_name,
    "selection_metric": "validation_macro_f1",
    "validation_metrics": selected_val_metrics,
    "final_test_metrics": test_metrics,
    "test_used_for_selection": False,
    "cv": "StratifiedKFold(n_splits=3, shuffle=True, random_state=42)",
    "search_sample_weight_used": True,
    "full_train_sample_weight_used": True,
    "best_search_params": search.best_params_,
    "best_cv_macro_f1": float(search.best_score_),
    "search_seconds": float(search_seconds),
    "final_train_seconds": float(full_train_seconds),
    "semantic_attack_mapping_verified": not any(
        str(v).startswith("Attack_") for v in label_mapping.values()
    ),
}
(M / "best_model_info.json").write_text(
    json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(info, ensure_ascii=False, indent=2))

{
  "methodology_version": 2,
  "selected_model": "XGBoost (Baseline retained)",
  "selection_metric": "validation_macro_f1",
  "validation_metrics": {
    "Accuracy": 0.9997269708739575,
    "Precision_macro": 0.9164416278835646,
    "Recall_macro": 0.9897998570638329,
    "F1_macro": 0.9366592369526422
  },
  "final_test_metrics": {
    "Accuracy": 0.9997974300032587,
    "Precision_macro": 0.8588411612594951,
    "Recall_macro": 0.9696598772124667,
    "F1_macro": 0.8967175034918179
  },
  "test_used_for_selection": false,
  "cv": "StratifiedKFold(n_splits=3, shuffle=True, random_state=42)",
  "search_sample_weight_used": true,
  "full_train_sample_weight_used": true,
  "best_search_params": {
    "subsample": 0.8,
    "reg_lambda": 5.0,
    "n_estimators": 150,
    "min_child_weight": 3,
    "max_depth": 4,
    "learning_rate": 0.08,
    "colsample_bytree": 1.0
  },
  "best_cv_macro_f1": 0.8782070007866482,
  "search_seconds": 1112.1346507072449,
  "final_train_seconds": 171.022121

In [8]:
# Build a final summary while keeping incomparable tasks explicit.
sup2 = pd.read_csv(M / "supervised_comparison.csv").copy()
sup2["Task"] = "multiclass_classification"
sup2["Primary_Metric"] = "macro_f1_validation"
sup2["Primary_Score"] = sup2["F1_macro"]

cnn2 = pd.read_csv(M / "cnn_comparison.csv").copy()
cnn2["Task"] = "multiclass_classification"
cnn2["Primary_Metric"] = "macro_f1_validation"
cnn2["Primary_Score"] = cnn2["F1_macro"]

unsup = pd.read_csv(M / "unsupervised_comparison.csv").copy()
unsup["Model"] = unsup["Algorithm"]
unsup["Primary_Metric"] = "binary_f1_validation"
unsup["Primary_Score"] = unsup["F1"]

final_summary = pd.concat([
    sup2[["Model","Task","Evaluation_Split","Accuracy","Primary_Metric","Primary_Score"]],
    cnn2[["Model","Task","Evaluation_Split","Accuracy","Primary_Metric","Primary_Score"]],
    unsup[["Model","Task","Evaluation_Split","Accuracy","Primary_Metric","Primary_Score"]],
], ignore_index=True)

final_summary.to_csv(M / "final_algorithms_summary.csv", index=False)
display(final_summary)

,Model,Task,Evaluation_Split,Accuracy,Primary_Metric,Primary_Score
0,Gradient Boosting (XGBoost),multiclass_classification,validation,0.999727,macro_f1_validation,0.936659
1,Random Forest,multiclass_classification,validation,0.999498,macro_f1_validation,0.899943
2,Logistic Regression,multiclass_classification,validation,0.934147,macro_f1_validation,0.699078
3,CNN (1D Conv),multiclass_classification,validation,0.984561,macro_f1_validation,0.672114
4,K-Means,binary_anomaly_detection,validation,0.585683,binary_f1_validation,0.184290
5,Isolation Forest,binary_anomaly_detection,validation,0.685074,binary_f1_validation,0.011008
6,DBSCAN,binary_anomaly_detection,validation,0.734600,binary_f1_validation,0.009455


## ما الذي نستطيع قوله في التقرير؟

- تم تدريب/تقييم 7 خوارزميات.
- Supervised + CNN تنفذ multiclass classification وتُقارن بـMacro-F1 على Validation.
- Unsupervised تنفذ binary anomaly detection وتُعرض مقاييسها منفصلة لأنها ليست نفس المهمة.
- أفضل production classifier اختير على Validation/CV.
- Test النهائي لم يدخل في قرار الاختيار أو الـtuning.